# DisloCluster — irradiation hardening by dislocation dynamics

Takes the immobile loop population a coupled march produced, rebuilds it as
**discrete loops in a periodic cube**, drives a dislocation through it under a
ramped resolved shear, and measures the CRSS increment Δτ(dose).

The design is
[`Docs/DisloCluster Manual/architecture/irradiation_hardening_dd_plan.md`](../Docs/DisloCluster%20Manual/architecture/irradiation_hardening_dd_plan.md);
what running it corrected and measured is
[`irradiation_hardening_implementation.md`](../Docs/DisloCluster%20Manual/architecture/irradiation_hardening_implementation.md)
beside it.

Everything below is driven from the six dicts in §2. Everything a run produces —
the staged DD cases, the figures, the snapshots, the movies and the provenance —
goes into **one timestamped directory** under `Simulations/output/`.

| route | what it gives | cost |
|---|---|---|
| **A** (this notebook) | Δτ at each dose, from a stress ramp on frozen loops | minutes to hours per dose |
| **C** | the dispersed-barrier law, α fitted on the route-A points | free, every dose |
| **D** | the same cell seeded by density instead of per-loop export | the smoke test |

**Two numbers are assumed, not measured**: the obstacle strengths α before
route A has fitted them, and the Taylor factor M that turns Δτ into Δσ_y.


## 1. Environment

In [1]:
# Every location resolves from the .dislocluster_root marker, so this notebook
# works from any checkout on any machine.
from pathlib import Path

import numpy as np

from dislocluster_code import paths, build
from dislocluster_code.studies import hardening as H
from dislocluster_code.post import dd_frames

print(paths.describe())

OK   repo root           D:\GitHub\DisloCluster
OK   venv                D:\GitHub\DisloCluster\.DisloClusterVenv
OK   0-D code            D:\GitHub\DisloCluster\ZrMicro
OK   0-D solver          D:\GitHub\DisloCluster\ZrMicro\build\Release\solver.exe
OK   input               D:\GitHub\DisloCluster\Simulations\input
OK   output (writes)     D:\GitHub\DisloCluster\Simulations\output
OK   output (also read)  D:\GitHub\DisloCluster\ZrMicro\output
OK   MoDELib root        D:\GitHub\DisloCluster\MoDELib3
OK   MoDELib build       D:\GitHub\DisloCluster\MoDELib3\build
OK   MoDELib DDomp       D:\GitHub\DisloCluster\MoDELib3\build\tools\DDomp\DDomp
OK   Zr material         D:\GitHub\DisloCluster\MoDELib3\Library\Materials\Zr3d_ghoniem.txt
OK   coupled sim case    D:\GitHub\DisloCluster\MoDELib3\tutorials\zrmicro_coupled
OK   sim case root       D:\GitHub\DisloCluster\MoDELib3\tutorials
OK   Simulations         D:\GitHub\DisloCluster\Simulations
OK   Gmsh generator      D:\GitHub\DisloCluster\Gm

In [2]:
# Builds either C++ code if it is missing, then RUNS both binaries -- being on
# disk is not the same as being runnable. Raises with actionable text otherwise.
build.preflight()

DisloCluster preflight
OK   repo root           D:\GitHub\DisloCluster
OK   venv                D:\GitHub\DisloCluster\.DisloClusterVenv
OK   0-D code            D:\GitHub\DisloCluster\ZrMicro
OK   0-D solver          D:\GitHub\DisloCluster\ZrMicro\build\Release\solver.exe
OK   input               D:\GitHub\DisloCluster\Simulations\input
OK   output (writes)     D:\GitHub\DisloCluster\Simulations\output
OK   output (also read)  D:\GitHub\DisloCluster\ZrMicro\output
OK   MoDELib root        D:\GitHub\DisloCluster\MoDELib3
OK   MoDELib build       D:\GitHub\DisloCluster\MoDELib3\build
OK   MoDELib DDomp       D:\GitHub\DisloCluster\MoDELib3\build\tools\DDomp\DDomp
OK   Zr material         D:\GitHub\DisloCluster\MoDELib3\Library\Materials\Zr3d_ghoniem.txt
OK   coupled sim case    D:\GitHub\DisloCluster\MoDELib3\tutorials\zrmicro_coupled
OK   sim case root       D:\GitHub\DisloCluster\MoDELib3\tutorials
OK   Simulations         D:\GitHub\DisloCluster\Simulations
OK   Gmsh generator      D:

[ZrMicro] solver runs : D:\GitHub\DisloCluster\ZrMicro\build\Release\solver.exe


[MoDELib] DDomp runs  : D:\GitHub\DisloCluster\MoDELib3\build\tools\DDomp\DDomp

ready.


{'solver': 'D:\\GitHub\\DisloCluster\\ZrMicro\\build\\Release\\solver.exe',
 'ddomp': 'D:\\GitHub\\DisloCluster\\MoDELib3\\build\\tools\\DDomp\\DDomp',
 'drift': []}

## 2. The control dicts

Nothing outside this section should need editing for a normal run.

In [3]:
MATERIAL = {
    "file":          "Zr3d_ghoniem.txt",   # MoDELib3/Library/Materials/
    "temperature_K": 573.0,
}

In [4]:
SOURCE = {
    # Where the loop population comes from.
    #
    #   run:  a coupled-march run directory -- the interior-quartile state is
    #         recomputed from its march_state.npz, which is the real thing.
    #   None: the plan's section-2 table, carried in hardening.PLAN_STATE_TABLE
    #         as a labelled CACHE for when that march is on another machine.
    # Resolved through `paths.find_runs()`, never as a literal path: the
    # notebook executes with `Simulations/` as its working directory, so a
    # repo-relative string resolves to Simulations/Simulations/... and the
    # run roots that are SEARCHED are not the one that is WRITTEN to.
    "run":    next(r for r in paths.find_runs() if r.name ==
                   "20260817_075527_563ab7d_500nmHex_Matched_SelfConsistent"),

    # Which doses to run route A at. The plan's choice is 1e-2, 0.1 and 10 dpa:
    # two parameters (alpha_c, alpha_a) need two points and three over-determine
    # them. Route C then covers every dose for free.
    "doses":  [1e-2, 0.1, 10.0],

    # Interior cut: nodes with distance-to-surface > frac x the maximum. The
    # boundary shell is a sink for mobile defects but NOT for loops, so a domain
    # mean measures the shell rather than the material.
    "frac":   0.5,
}

In [5]:
CELL = {
    # Cube edge. The box edges are made LATTICE VECTORS of hcp Zr, so the
    # periodic shifts map glide planes onto themselves -- 500 nm becomes
    # (n,m,p) = (1548, 894, 971) and 500.00 x 500.15 x 500.06 nm.
    #
    # 200 nm is what a laptop can do; 500 nm is the smallest defensible cell at
    # high dose (r_<c> reaches 55 nm) and costs ~3 s/step on 8 cores.
    "L_nm":      300.0,

    # Loop placement is a random draw. MoDELib refuses a few realizations per
    # thousand (a network node lands just outside the primary cell); the
    # campaign redraws with the next seed and records which one it used.
    "seed":      0,
    "max_seeds": 6,
}

In [6]:
LOAD = {
    # Stress ramp on the prismatic <a> system. None -> 1 MPa per 1e4 b/cs,
    # which is 6.97e8 MPa/s here: fast in SI, quasi-static in DD terms because
    # a pinned line relaxes in ~1e2 b/cs.
    "tau_dot_MPa_s":  None,

    # Steps and output cadence. Every output step is a movie frame and a
    # possible snapshot, so this also sets the temporal resolution of §7.
    "n_steps":        8000,
    "output_every":   50,

    # The saving that makes this affordable: a frozen loop's segments fall into
    # the largest subcycling bin and are re-evaluated once per 100 steps, while
    # still contributing stress every step. Measured 5.4x on the 200 nm cell.
    "subcycling":     True,

    # The offset criterion on the tau-gamma_p curve. THIS IS A CHOICE AND IT
    # MATTERS: an irradiated cell creeps below its breakaway stress, so 1e-4
    # lands in the creep tail and 1e-2 after general flow. Every offset in
    # hardening.OFFSETS is reported whatever this is set to.
    "offset":         1e-3,

    # The mobile source: one periodic dipole on the loaded slip system. Not a
    # density-seeded population -- that generator picks the slip system itself,
    # and a basal choice has exactly zero resolved shear under this load.
    "dipole_height":  200,
}

In [7]:
OUTPUT = {
    "tag":            "hardening_300nmHex",   # goes into the run-directory name
    "figures":        True,
    "snapshots":      6,      # configurations rendered per dose (0 disables)
    "movies":         True,   # one movie per dose
    "movie_fps":      8,
    "movie_frames":   60,     # cap; frames are SOLVED steps, never interpolated

    # Reporting only -- neither is measured by this notebook.
    "taylor_M":       3.0,
    "alpha_assumed":  (0.4, 0.2),   # the plan's estimate, drawn for comparison
}

## 3. Resolve, and look at the state before paying for anything

The interior state and the cell are cheap to compute and are what the whole
campaign is built on, so they are printed before a single DD step runs.

In [8]:
mat = H.Material.from_file(MATERIAL["file"], MATERIAL["temperature_K"])
box = H.commensurate_box(CELL["L_nm"], mat)
states = H.states_for(SOURCE["run"], mat, SOURCE["frac"])

print(mat.summary())
print()
print("cell:", box.describe())
print()
print(f"source: {states[0]['run']}")
print(H.format_state_table(states))

Zr3d_ghoniem.txt at 573 K: b=0.323 nm, c/a=1.5944272, mu=33 GPa, cs=2250 m/s, b/cs=1.436e-13 s, Omega=2.326553e-29 m^3 (0.6904 b^3)

cell: (n,m,p) = (929, 536, 583)  F = diag(929.0, 928.4, 929.6) b  = 300.07 x 299.87 x 300.24 nm  V = 2.702e-20 m^3

source: D:\GitHub\DisloCluster\Simulations\output\20260817_075527_563ab7d_500nmHex_Matched_SelfConsistent
     dpa  N_c [m^-3]  r_c [nm]  N_a [m^-3]  r_a [nm]  rho_c [m^-2]  rho_a [m^-2]
       0     4.3e+08      0.17    1.29e+09     0.151         0.458          1.23
  0.0001    5.19e+19       3.4    4.81e+19      2.09      1.11e+12      6.32e+11
   0.001    3.87e+20      3.52    4.73e+20      2.13      8.57e+12      6.32e+12
    0.01     7.7e+20      5.43    3.83e+21      2.51      2.63e+13      6.03e+13
     0.1    7.68e+20      25.8    1.09e+22      2.19      1.25e+14       1.5e+14
       1    7.62e+20      57.1     6.2e+21      2.96      2.74e+14      1.15e+14
       2    7.63e+20      55.9    4.12e+21      3.62      2.68e+14      9.37e+

In [9]:
# What the cube will actually hold at each requested dose, before it is built.
# `2r/d` is the overlap ratio: at 1.0 the loops are geometrically touching and
# the continuum population is outside its own validated range.
for d in SOURCE["doses"]:
    st = min(states, key=lambda s: abs(s["dose"] - d))
    H.cube_population(box, mat, state=st, seed=CELL["seed"], verbose=True)

  cube population from state at 0.01 dpa, 300 nm cube
    fam       N    r[nm]    d[nm]    2r/d     seg  flag
    c        21     5.43    108.8   0.100     252  
    a1       34     2.51     92.6   0.054     544  
    a2       34     2.51     92.6   0.054     544  
    a3       34     2.51     92.6   0.054     544  
  cube population from state at 0.1 dpa, 300 nm cube
    fam       N    r[nm]    d[nm]    2r/d     seg  flag
    c        21    25.84    108.8   0.475     252  
    a1       98     2.19     65.1   0.067    1568  
    a2       98     2.19     65.1   0.067    1568  
    a3       98     2.19     65.1   0.067    1568  
  cube population from state at 10 dpa, 300 nm cube
    fam       N    r[nm]    d[nm]    2r/d     seg  flag
    c        21    55.82    108.8   1.027     252  
    a1       37     3.64     90.0   0.081     592  
    a2       37     3.64     90.0   0.081     592  
    a3       37     3.64     90.0   0.081     592  


## 4. The run directory

One timestamped directory holds everything: the staged DD cases, the figures,
the snapshots, the movies and the provenance.

In [10]:
run_dir = paths.run_dir(OUTPUT["tag"])
cases   = run_dir / "cases"
cases.mkdir(parents=True, exist_ok=True)
print(run_dir)

D:\GitHub\DisloCluster\Simulations\output\20260821_174142_b5c7860_hardening_300nmHex


## 5. The campaign

One control (same cell, same source, **no loops**) and one case per dose. Δτ is
a difference: without the control there is no measurement, because the absolute
stress carries the Peierls law and the source geometry.

This is the expensive cell. On the 200 nm cube at 8000 steps it is a few minutes
per case; at 500 nm expect ~3 s/step.

In [11]:
camp = H.campaign(
    cases / "hard", SOURCE["doses"],
    L_nm=CELL["L_nm"], run_dir=SOURCE["run"], material=mat,
    tau_dot_MPa_s=LOAD["tau_dot_MPa_s"], n_steps=LOAD["n_steps"],
    output_frequency=LOAD["output_every"], seed=CELL["seed"],
    max_seeds=CELL["max_seeds"], offset=LOAD["offset"],
    dipole_height=LOAD["dipole_height"], subcycling=LOAD["subcycling"],
    tag="dose")

campaign: (n,m,p) = (929, 536, 583)  F = diag(929.0, 928.4, 929.6) b  = 300.07 x 299.87 x 300.24 nm  V = 2.702e-20 m^3
  3 doses + 1 control, 8000 steps each


staged D:\GitHub\DisloCluster\Simulations\output\20260821_174142_b5c7860_hardening_300nmHex\cases\hard_control
  route A, (n,m,p) = (929, 536, 583)  F = diag(929.0, 928.4, 929.6) b  = 300.07 x 299.87 x 300.24 nm  V = 2.702e-20 m^3
  loops: NONE (control run)
  ramp 6.965e+08 MPa/s = 3.03e-09 mu per b/cs = 1 MPa per 1e+04 b/cs
  resolved on s=[1.0, 0.0, 0.0] n=[0.0, -1.0, 0.0]
  generating microstructure in hard_control


  DDomp in hard_control


    8000 steps in 178 s = 0.0222 s/step (exit 0)
  hard_control: tau(0.001) = 8.706 MPa [1e-05:0.08 0.0001:0.83 0.001:8.71 0.01:19.54], tau_max = 53.813 MPa, 160 output steps
  cube population from field at 0.01 dpa, 300 nm cube
    fam       N    r[nm]    d[nm]    2r/d     seg  flag
    c        21     5.45    108.8   0.100     252  
    a1       34     2.31     92.6   0.050     544  
    a2       34     2.51     92.6   0.054     544  
    a3       34     2.31     92.6   0.050     544  


staged D:\GitHub\DisloCluster\Simulations\output\20260821_174142_b5c7860_hardening_300nmHex\cases\hard_dose_0.01dpa_s0
  route A, (n,m,p) = (929, 536, 583)  F = diag(929.0, 928.4, 929.6) b  = 300.07 x 299.87 x 300.24 nm  V = 2.702e-20 m^3
  loops: {'c': 21, 'a1': 34, 'a2': 34, 'a3': 34, 'total': 123}
  ramp 6.965e+08 MPa/s = 3.03e-09 mu per b/cs = 1 MPa per 1e+04 b/cs
  resolved on s=[1.0, 0.0, 0.0] n=[0.0, -1.0, 0.0]
  generating microstructure in hard_dose_0.01dpa_s0


...

    8000 steps in 42281 s = 5.29 s/step (exit 0)
  hard_dose_0.1dpa_s0: tau(0.001) = 22.087 MPa [1e-05:0.09 0.0001:0.87 0.001:22.09 0.01:35.37], tau_max = 40.678 MPa, 160 output steps
  cube population from field at 10 dpa, 300 nm cube
    fam       N    r[nm]    d[nm]    2r/d     seg  flag
    c        21    55.92    108.8   1.028     252  
    a1       37     3.71     90.0   0.082     592  
    a2       37     3.54     90.0   0.079     592  
    a3       37     3.48     90.0   0.077     592  


staged D:\GitHub\DisloCluster\Simulations\output\20260821_174142_b5c7860_hardening_300nmHex\cases\hard_dose_10dpa_s0
  route A, (n,m,p) = (929, 536, 583)  F = diag(929.0, 928.4, 929.6) b  = 300.07 x 299.87 x 300.24 nm  V = 2.702e-20 m^3
  loops: {'c': 21, 'a1': 37, 'a2': 37, 'a3': 37, 'total': 132}
  ramp 6.965e+08 MPa/s = 3.03e-09 mu per b/cs = 1 MPa per 1e+04 b/cs
  resolved on s=[1.0, 0.0, 0.0] n=[0.0, -1.0, 0.0]
  generating microstructure in hard_dose_10dpa_s0


  DDomp in hard_dose_10dpa_s0


    8000 steps in 2896 s = 0.362 s/step (exit 0)
  hard_dose_10dpa_s0: tau(0.001) = 29.118 MPa [1e-05:0.06 0.0001:0.60 0.001:29.12 0.01:nan], tau_max = 36.802 MPa, 160 output steps



  fitted alpha_c = 0.208, alpha_a = 0.020  (rms 0.38 MPa)


## 6. Results

In [12]:
ctrl = camp["control"]
print(f"control (no loops):  tau({LOAD['offset']:g}) = "
      f"{ctrl['tau_offset_MPa']:.3f} MPa\n")
print(f"{'dpa':>8} {'seed':>5} {'obst':>6} {'tau_off':>9} {'dtau':>9} "
      f"{'dsigma_y':>9} {'s/step':>8}")
for r in camp["runs"]:
    print(f"{r['dose']:>8.4g} {r['seed']:>5d} {H.obstacle_count(r):>6d} "
          f"{r['tau_offset_MPa']:>9.3f} {r['delta_tau_MPa']:>9.3f} "
          f"{r['delta_tau_MPa'] * OUTPUT['taylor_M']:>9.1f} "
          f"{r['s_per_step']:>8.4g}")

a = camp.get("alpha")
if a:
    print(f"\nfitted   alpha_c = {a['alpha_c']:.3f}   alpha_a = {a['alpha_a']:.3f}"
          f"   (rms {a['rms_MPa']:.2f} MPa)")
    print(f"assumed  alpha_c = {OUTPUT['alpha_assumed'][0]:g}   "
          f"alpha_a = {OUTPUT['alpha_assumed'][1]:g}   (the plan's estimate)")

control (no loops):  tau(0.001) = 8.706 MPa

     dpa  seed   obst   tau_off      dtau  dsigma_y   s/step
    0.01     0    123    15.255     6.549      19.6   0.5283
     0.1     0    315    22.087    13.381      40.1    5.285
      10     0    132    29.118    20.412      61.2    0.362

fitted   alpha_c = 0.208   alpha_a = 0.020   (rms 0.38 MPa)
assumed  alpha_c = 0.4   alpha_a = 0.2   (the plan's estimate)


In [13]:
# The offset sensitivity, stated rather than buried: the same three runs at
# every criterion. If these columns disagree by more than the scatter you are
# willing to quote, the number to quote is a range.
print(f"{'dpa':>8} " + " ".join(f"{'d@' + k:>10}" for k in ctrl["tau_at_offsets"]))
for r in camp["runs"]:
    print(f"{r['dose']:>8.4g} " + " ".join(
        f"{r['tau_at_offsets'][k] - ctrl['tau_at_offsets'][k]:>10.2f}"
        for k in ctrl["tau_at_offsets"]))

     dpa    d@1e-05   d@0.0001    d@0.001     d@0.01
    0.01       2.05       1.74       6.55       8.32
     0.1       0.00       0.04      13.38      15.83
      10      -0.02      -0.23      20.41        nan


## 7. Route C at every dose

The dispersed-barrier law with the α just fitted, applied to every dose the
march produced — including doses no DD case was run at. This is the deliverable
the plan is built around: three DD campaigns, not nine.

In [14]:
alpha = ({"c": camp["alpha"]["alpha_c"], "a": camp["alpha"]["alpha_a"]}
         if camp.get("alpha") else
         {"c": OUTPUT["alpha_assumed"][0], "a": OUTPUT["alpha_assumed"][1]})
rows = [H.dispersed_barrier(s, alpha, mat, taylor_M=OUTPUT["taylor_M"])
        for s in states]
print(f"alpha_c = {alpha['c']:.3f}, alpha_a = {alpha['a']:.3f}, "
      f"M = {OUTPUT['taylor_M']:g}")
print(H.format_routec(rows, OUTPUT["taylor_M"]))

alpha_c = 0.208, alpha_a = 0.020, M = 3
     dpa    dtau_c    dtau_a      dtau   dsigma_y
             [MPa]     [MPa]     [MPa]      [MPa]
       0       0.0       0.0       0.0        0.0
  0.0001       1.3       0.1       1.3        4.0
   0.001       3.7       0.3       3.7       11.0
    0.01       6.4       0.9       6.5       19.4
     0.1      14.0       1.5      14.0       42.1
       1      20.7       1.3      20.7       62.1
       2      20.5       1.2      20.5       61.5
       5      20.4       1.2      20.5       61.4
      10      20.4       1.2      20.5       61.4


## 8. Figures

`flow_curves.png` is the measurement itself — the shape says whether the number
means anything. `delta_tau_vs_dose.png` puts the DD anchors against both laws.
`loops_*.png` is the obstacle field the cell was given.

In [15]:
if OUTPUT["figures"]:
    figs = H.figures(cases / "hard_campaign.json", out_dir=run_dir / "figures",
                     material=mat, run_dir=SOURCE["run"],
                     taylor_M=OUTPUT["taylor_M"])

  wrote D:\GitHub\DisloCluster\Simulations\output\20260821_174142_b5c7860_hardening_300nmHex\figures\flow_curves.png
  wrote D:\GitHub\DisloCluster\Simulations\output\20260821_174142_b5c7860_hardening_300nmHex\figures\delta_tau_vs_dose.png
  wrote D:\GitHub\DisloCluster\Simulations\output\20260821_174142_b5c7860_hardening_300nmHex\figures\loops_0.01dpa.png
  wrote D:\GitHub\DisloCluster\Simulations\output\20260821_174142_b5c7860_hardening_300nmHex\figures\loops_0.1dpa.png
  wrote D:\GitHub\DisloCluster\Simulations\output\20260821_174142_b5c7860_hardening_300nmHex\figures\loops_10dpa.png


## 9. Snapshots and movies of the dislocation in the obstacle field

Each frame is a **solved configuration** — one of the `evl_<N>.txt` files the run
wrote — not an interpolant. Two panels: the cell in 3-D, and everything projected
along the glide-plane normal, which is where pinning, bowing between obstacles
and the released arc after breakaway are actually visible.

Red is the gliding dislocation (`GLISSILELOOP`), blue the ⟨c⟩ basal loops and
grey the ⟨a⟩ prismatic loops — all frozen, because they are `SESSILELOOP`s and
`climbSolverType=none`.

One movie per dose. MP4 needs ffmpeg (`pip install imageio-ffmpeg`); without it
the GIF is written and nothing fails.

In [16]:
snap_dir = run_dir / "snapshots"
for r in camp["runs"]:
    tag = f"{r['dose']:g}dpa".replace(".", "p")
    if OUTPUT["snapshots"]:
        dd_frames.snapshots(r["sim_dir"], out_dir=snap_dir / tag,
                            n=OUTPUT["snapshots"], dose=r["dose"])

# The control too: the same cell with nothing in it, which is what the
# increment is measured against.
if OUTPUT["snapshots"]:
    dd_frames.snapshots(ctrl["sim_dir"], out_dir=snap_dir / "control",
                        n=OUTPUT["snapshots"])

  6 snapshots -> D:\GitHub\DisloCluster\Simulations\output\20260821_174142_b5c7860_hardening_300nmHex\snapshots\0p01dpa


  6 snapshots -> D:\GitHub\DisloCluster\Simulations\output\20260821_174142_b5c7860_hardening_300nmHex\snapshots\0p1dpa


  6 snapshots -> D:\GitHub\DisloCluster\Simulations\output\20260821_174142_b5c7860_hardening_300nmHex\snapshots\10dpa


  6 snapshots -> D:\GitHub\DisloCluster\Simulations\output\20260821_174142_b5c7860_hardening_300nmHex\snapshots\control


In [17]:
if OUTPUT["movies"]:
    mov_dir = run_dir / "movies"
    for r in camp["runs"]:
        tag = f"{r['dose']:g}dpa".replace(".", "p")
        print(f"{r['dose']:g} dpa:")
        dd_frames.movie(r["sim_dir"], out_file=mov_dir / f"motion_{tag}",
                        fps=OUTPUT["movie_fps"],
                        max_frames=OUTPUT["movie_frames"])

0.01 dpa:


  motion_0p01dpa.mp4  (60 frames, 8 fps, 0.5 MB)
  motion_0p01dpa.gif  (60 frames, 8 fps, 2.1 MB)
0.1 dpa:


  motion_0p1dpa.mp4  (60 frames, 8 fps, 0.7 MB)
  motion_0p1dpa.gif  (60 frames, 8 fps, 2.5 MB)
10 dpa:


  motion_10dpa.mp4  (60 frames, 8 fps, 0.6 MB)
  motion_10dpa.gif  (60 frames, 8 fps, 2.4 MB)


## 10. Provenance

Everything needed to reproduce the run, beside its results.

In [18]:
import json, time

prov = dict(
    tag=OUTPUT["tag"], created=time.strftime("%Y-%m-%d %H:%M:%S"),
    git_hash=paths.git_hash(), material=MATERIAL, source=SOURCE, cell=CELL,
    load=LOAD, output={k: v for k, v in OUTPUT.items()},
    box=box.describe(), cell_volume_m3=box.volume_m3,
    control=dict(sim_dir=ctrl["sim_dir"],
                 tau_offset_MPa=ctrl["tau_offset_MPa"],
                 tau_at_offsets=ctrl["tau_at_offsets"]),
    runs=[dict(dose=r["dose"], seed=r["seed"], n_loops=r["n_loops"], n_obstacles=H.obstacle_count(r),
               tau_offset_MPa=r["tau_offset_MPa"],
               delta_tau_MPa=r["delta_tau_MPa"],
               tau_at_offsets=r["tau_at_offsets"],
               s_per_step=r["s_per_step"], sim_dir=r["sim_dir"])
          for r in camp["runs"]],
    alpha=camp.get("alpha"),
    route_c=[{k: v for k, v in row.items() if k != "per_family"} for row in rows],
)
(run_dir / "provenance.json").write_text(json.dumps(prov, indent=2, default=str),
                                         encoding="utf-8")

(run_dir / "provenance.md").write_text("\n".join([
    f"# Irradiation hardening — {OUTPUT['tag']}",
    "",
    f"- created  {prov['created']}",
    f"- git      {prov['git_hash']}",
    f"- material {MATERIAL['file']} at {MATERIAL['temperature_K']:g} K",
    f"- cell     {box.describe()}",
    f"- source   {states[0]['run']}",
    f"- load     ramp {'default (1 MPa per 1e4 b/cs)' if LOAD['tau_dot_MPa_s'] is None else LOAD['tau_dot_MPa_s']}"
    f", {LOAD['n_steps']} steps, offset {LOAD['offset']:g}",
    "",
    "| dpa | seed | obstacles | tau_off [MPa] | dtau [MPa] |",
    "|---:|---:|---:|---:|---:|",
] + [f"| {r['dose']:g} | {r['seed']} | {H.obstacle_count(r)} | "
     f"{r['tau_offset_MPa']:.3f} | {r['delta_tau_MPa']:.3f} |"
     for r in camp["runs"]] + [
    "",
    "**Frozen loops, one realization per dose, and alpha and M are assumptions.**",
    "See the implementation note for what that costs.",
]), encoding="utf-8")

print(run_dir)
for p in sorted(run_dir.rglob("*")):
    if p.is_file() and p.suffix in (".png", ".gif", ".mp4", ".md", ".json"):
        print(f"  {p.relative_to(run_dir)}")

D:\GitHub\DisloCluster\Simulations\output\20260821_174142_b5c7860_hardening_300nmHex
  cases\hard_campaign.json
  cases\hard_control\hardening.json
  cases\hard_dose_0.01dpa_s0\hardening.json
  cases\hard_dose_0.1dpa_s0\hardening.json
  cases\hard_dose_10dpa_s0\hardening.json
  figures\delta_tau_vs_dose.png
  figures\flow_curves.png
  figures\loops_0.01dpa.png
  figures\loops_0.1dpa.png
  figures\loops_10dpa.png
  movies\motion_0p01dpa.gif
  movies\motion_0p01dpa.mp4
  movies\motion_0p1dpa.gif
  movies\motion_0p1dpa.mp4
  movies\motion_10dpa.gif
  movies\motion_10dpa.mp4
  provenance.json
  provenance.md
  snapshots\0p01dpa\frame_000000.png
  snapshots\0p01dpa\frame_001600.png
  snapshots\0p01dpa\frame_003200.png
  snapshots\0p01dpa\frame_004750.png
  snapshots\0p01dpa\frame_006350.png
  snapshots\0p01dpa\frame_007950.png
  snapshots\0p1dpa\frame_000000.png
  snapshots\0p1dpa\frame_001600.png
  snapshots\0p1dpa\frame_003200.png
  snapshots\0p1dpa\frame_004750.png
  snapshots\0p1dpa\fra